# Dokumentacja Projektu
W tym notatniku znajduje się dokumentacja i przykłady użycia modeli do zadań Super Resolution oraz Denoising.

Poniższy kod demonstruje w jaki sposób wytrenować modele dla obu zadań oraz jak obliczyć metryki jakości wynikowych obrazów.

Wyniki z obliczeń znajdują się w folderach outputs/[zadanie]_[metoda optymalizacji]_[współczynnik uczenia]

Kod służący do ewaluacji modelu:

In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../classes'))

from evaluate_model import evaluate_all_models

evaluate_all_models()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: d:\studia\Studia 2 stopnia\Semestr 3\SIGK\SIGK-projects\.venv\Lib\site-packages\lpips\weights\v0.1\vgg.pth
Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]
Loading model from: d:\studia\Studia 2 stopnia\Semestr 3\SIGK\SIGK-projects\.venv\Lib\site-packages\lpips\weights\v0.1\vgg.pth

Evaluating best model for super_resolution | Criterion: CombinedPerceptualLoss | LR: 0.0001 | Model: BetterUNet | Epoch: 20
Ocenianie na: cuda
Rozpoczynam ewaluację 100 obrazów...

Evaluating best model for super_resolution | Criterion: L1Loss | LR: 0.0001 | Model: BetterUNet | Epoch: 20
Ocenianie na: cuda
Rozpoczynam ewaluację 100 obrazów...

Evaluating best model for super_resolution | Criterion: L1Loss | LR: 0.0005 | Model: BetterUNet | Epoch: 20
Ocenianie na: cuda
Rozpoczynam ewaluację 100 obrazów...

Evaluating best model for super_resolution | Criterion: MSELoss | LR: 0.0001 | Model: Better

## Ewaluacja modelu
W tej sekcji załadujemy wytrenowany wcześniej model z folderu `outputs` i sprawdzimy jego wyniki na zbiorze walidacyjnym, obliczając przy tym metryki (PSNR, SSIM, LPIPS) oraz wizualizując wyniki.

In [2]:
import pandas as pd

csv_file = "../outputs/eval_results/all_metrics.csv"

df_results = pd.read_csv(csv_file)

df_sorted = df_results.sort_values(by="PSNR", ascending=False)

display(df_sorted)

best_model_info = df_sorted.iloc[0]
print(f"Best model: {best_model_info['Model File']}")
print(f"Trained with Criterion: {best_model_info['Criterion']} and LR: {best_model_info['Learning Rate']}")

,Task,Criterion,Learning Rate,Model File,Samples,PSNR,SSIM,LPIPS
7,super_resolution,L1Loss,0.0005,ResNetRestoration_model_epoch_20_super_resolut...,100,19.608329,0.503481,0.533365
0,super_resolution,CombinedPerceptualLoss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.1...,100,19.572103,0.507953,0.408026
9,super_resolution,MSELoss,0.0005,ResNetRestoration_model_epoch_20_super_resolut...,100,19.531339,0.498062,0.532824
2,super_resolution,L1Loss,0.0005,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.493334,0.500538,0.531630
8,super_resolution,MSELoss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.452851,0.491374,0.534597
3,super_resolution,MSELoss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.433840,0.487794,0.528986
12,super_resolution,L1Loss,0.0005,SimpleUNet_model_epoch_20_super_resolution_0.0...,100,19.418272,0.496917,0.519769
6,super_resolution,L1Loss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.356983,0.486303,0.543275
1,super_resolution,L1Loss,0.0001,BetterUNet_model_epoch_20_super_resolution_0.0...,100,19.347359,0.489079,0.533489
5,super_resolution,CombinedPerceptualLoss,0.0001,ResNetRestoration_model_epoch_20_super_resolut...,100,19.311995,0.491358,0.513040


Best model: ResNetRestoration_model_epoch_20_super_resolution_0.0707.pth
Trained with Criterion: L1Loss and LR: 0.0005


## Ewaluacja rozwiązań bazowych (Baselines)
Zgodnie z wymaganiami projektu, musimy porównać nasze modele z tradycyjnymi algorytmami:
1. **Zwiększanie rozdzielczości (Super-Resolution):** Interpolacja bikubiczna (OpenCV `resize`).
2. **Odszumianie (Denoising):** Filtracja bilateralna (`denoise_bilateral` z biblioteki `skimage`).

Poniższy kod iteruje przez zbiór walidacyjny, przetwarza obrazy za pomocą metod bazowych i wylicza uśrednione metryki PSNR, SSIM oraz LPIPS, a następnie prezentuje je w tabeli.

In [3]:
from services import evaluate_baselines

df_baselines = evaluate_baselines()

print("\n--- Tabela wyników metod bazowych ---")
display(df_baselines)

Evaluation: OpenCV Bicubic Interpolation (Super-Resolution)...
Evaluation: skimage denoise_bilateral (Denoising)...

--- Tabela wyników metod bazowych ---


,Metoda,PSNR,SSIM,LPIPS
0,bicubic_interpolation_super_resolution,19.3285,0.4846,0.5288
1,denoise_bilateral_skimage,27.4192,0.9121,0.1506
